# IMDB Vector Search using Milvus Client

首先，导入一些常用库并定义数据读取函数。

In [1]:
import sys, time, pprint
import pandas as pd
import numpy as np
import os

from IPython.display import display

# Import custom functions for splitting and search
sys.path.append("..")
import milvus_utilities as _utils


In [2]:
CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')

# Start up a local Milvus Lite

STEP 1. CONNECT TO MILVUS

In [3]:
from pymilvus import MilvusClient

client = MilvusClient("../milvus_demo.db")
print(f"Milvus Lite 客户端已初始化，数据将保存在 ../milvus_demo.db")

Milvus Lite 客户端已初始化，数据将保存在 ../milvus_demo.db


## 加载嵌入模型的检查点，并使用它来生成向量嵌入

嵌入模型：我们将使用 HuggingFace 上提供的开源句子转换模型来编码文档文本。我们从 HuggingFace 下载模型，并在本地运行。

以下两个模型参数需要注意：

1. EMBEDDING_LENGTH 指的是嵌入向量的维度或长度。在此情况下，输入文本中每个标记生成的嵌入向量长度相同，均为 1024。这种嵌入大小通常与基于 BERT 的模型相关，这些嵌入用于下游任务，如分类、问答或文本生成。

2. MAX_SEQ_LENGTH 是编码器模型能够处理的最大输入序列长度。在这种情况下，如果输入的序列长度超过 512 个标记，超出部分将被（静默地！）截断。因此，需要采用分块策略，将输入文本分割成长度适配模型输入的若干小块。

STEP 2. DOWNLOAD AN OPEN SOURCE EMBEDDING MODEL.

In [4]:
import torch
from torch.nn import functional as F
from sentence_transformers import SentenceTransformer

# Initialize torch settings
torch.backends.cudnn.deterministic=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {DEVICE}")

device: cuda


Load the model from huggingface model hub

In [5]:
model_name="WhereIsAI/UAE-Large-V1"
encoder=SentenceTransformer(model_name,device=DEVICE)
print(type(encoder))
print(encoder)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>
SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
)


Get the model parameters and save for later

In [6]:
EMBEDDING_LENGTH=encoder.get_embedding_dimension()
MAX_SEQ_LENGTH_INTOKENS=encoder.get_max_seq_length()
# # Assume tokens are 3 characters long.
# MAX_SEQ_LENGTH = MAX_SEQ_LENGTH_IN_TOKENS * 3
# HF_EOS_TOKEN_LENGTH = 1 * 3
# Test with 512 sequence length.
MAX_SEQ_LENGTH=MAX_SEQ_LENGTH_INTOKENS
HF_EOS_TOKEN_LENGTH=1


Inspect model parameters

In [7]:
print(f"model name: {model_name}")
print(f"EMBEDDING_LENGTH: {EMBEDDING_LENGTH}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")

model name: WhereIsAI/UAE-Large-V1
EMBEDDING_LENGTH: 1024
MAX_SEQ_LENGTH: 512


## Create a Milvus collection
在 Milvus 中，你可以将集合类比为 SQL 数据库中的“表”。该集合将包含以下内容：

- Schema模型架构（或无架构的 Milvus 客户端）

    💡你需要从嵌入模型中获取向量的 EMBEDDING_LENGTH 参数。常见值如下：
    - sbert 嵌入模型：768
    - ada-002 OpenAI 嵌入模型：1536

- Vector index向量索引，用于高效向量搜索
- Vector distance metric向量距离度量，用于计算最近邻向量
- Consistency level一致性级别：Milvus 支持事务一致性，但根据 CAP 定理，必须牺牲一定的延迟。

    💡 由于电影评论搜索并非关键任务，因此最终一致性在此处是可接受的。

### Exercise #1

Create a collection named "movies". Use the default AUTOINDEX
> 💡 AUTOINDEX works on both Milvus and Zilliz Cloud (where it is the fastest!)

In [8]:
COLLECTION_NAME="movies"
client.drop_collection(COLLECTION_NAME)
client.create_collection(COLLECTION_NAME,EMBEDDING_LENGTH,params='AUTOINDEX')
print(client.describe_collection(COLLECTION_NAME))
print(f"Created collection: {COLLECTION_NAME}")

{'collection_name': 'movies', 'auto_id': False, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'is_primary': True}, {'field_id': 0, 'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'consistency_level_name': 'Strong', 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True, 'enable_namespace': False}
Created collection: movies


## Add a Vector Index

Vector Index决定了用于查找用户提交查询时数据中与之最接近的向量的**搜索算法**。

大多数Vector Index根据数据库的使用场景（插入向量或搜索向量）采用不同的参数组合：

- **插入向量**（创建模式）
- **搜索向量**（搜索模式）

请向下滚动文档页面，查看 Milvus 提供的不同Vector Index列表。例如：

- FLAT — 确定性穷尽搜索
- IVF_FLAT 或 IVF_SQ8 — 哈希索引（随机近似搜索）
- HNSW — 图形索引（随机近似搜索）
- AUTOINDEX — 根据 OSS 与 Zilliz 云、GPU 类型及数据规模自动确定

除了搜索算法外，我们还需要指定**距离度量**，即定义向量空间中“接近”的标准。在下方单元格中选择了 `HNSW` 搜索索引。其可用的距离度量包括以下几种：

- L2 — L2 范数
- IP — 点积
- COSINE — 角度距离

💡大多数应用场景更适合使用归一化嵌入（normalized embeddings），此时 L2 无用（每个向量长度为1），IP 和 COSINE 相等。仅当您计划保持嵌入未归一化时，才应选择 L2。

STEP 3. CREATE A NO-SCHEMA MILVUS COLLECTION AND DEFINE THE DATABASE INDEX.

In [9]:
# 对于 vector长度，使用嵌入模型中的嵌入长度
print(f"Embedding length: {EMBEDDING_LENGTH}")

# Set the Milvus collection name
COLLECTION_NAME="movies"

# Add custom HNSW search index to the collection
    # M = 每层最大图连接数。M越大，图的密度越高
    # M的选择范围：4~64，M越大，数据量和嵌入长度也越大
M=16
    # efConstruction = 每层的最近邻候选数
    # 使用经验法则：int. 8~512，efConstruction = M * 2
efConstruction=M*2

# Create the search index fro local Milvus server
INDEX_PARAMS=dict({
    'M':M,
    "efConstruction":efConstruction
})
index_params={
    "index_type":"HNSW",
    "metric_type":"COSINE",
    "params":INDEX_PARAMS
}

# Use no-schema Milvus client (flexible json key:value format)
mc=MilvusClient("../milvus_demo.db")

# Check if collection already exists, if so drop it
has=mc.has_collection(COLLECTION_NAME)
if has:
    drop_result=mc.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection: {COLLECTION_NAME}")

mc.create_collection(
    COLLECTION_NAME,
    EMBEDDING_LENGTH,
    consistency_level="Eventually",
    auto_id=True,
    overwrite=True,
    # skip setting params below, if using AUTOINDEX
    params=index_params
)

print(f"Created collection: {COLLECTION_NAME}")
print(mc.describe_collection(COLLECTION_NAME))

Embedding length: 1024
Successfully dropped collection: movies
Created collection: movies
{'collection_name': 'movies', 'auto_id': True, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 0, 'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'consistency_level_name': 'Strong', 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True, 'enable_namespace': False}


## Read CSV data into a pandas dataframe

本笔记本中使用的数据来自斯坦福人工智能实验室的[IMDB large movie review dataset](https://ai.stanford.edu/~amaas/data/sentiment/)。该数据集经过便捷处理，包含50,000条样本（正面与负面评论各占50%），并具有以下列：movie_index, raw review text, and movie rating

In [10]:
# 1. Download data from https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
# 2. Move .csv file to data/ folder.

# citation:  ACL 2011, @InProceedings{maas-EtAl:2011:ACL-HLT2011,
#   author    = {Maas, Andrew L.  and  Daly, Raymond E.  and  Pham, Peter T.  and  Huang, Dan  and  Ng, Andrew Y.  and  Potts, Christopher},
#   title     = {Learning Word Vectors for Sentiment Analysis},
#   booktitle = {Proceedings of the 49th Annual Meeting of the Association for Computational Linguistics: Human Language Technologies},
#   month     = {June},
#   year      = {2011},
#   address   = {Portland, Oregon, USA},
#   publisher = {Association for Computational Linguistics},
#   pages     = {142--150},
#   url       = {http://www.aclweb.org/anthology/P11-1015}
# }

In [11]:
# Read locally stored data
filepath="data/movie_data.csv"

df=pd.read_csv(f"{filepath}")

# Drop duplicateds
df.drop_duplicates(keep='first',inplace=True)

# Change label column names
df.columns=['text','label_int']

# Map numbers to text 'Positive' and 'Negative' for sentiment labels
df["label"]=df["label_int"].apply(_utils.sentiment_score_to_name)

# Split data into train/val/test
columns=['movie_index','text','label_int','label']
df,df_train,df_val,df_test=_utils.partition_dataset(df,columns,smoke_test=False)
print(f"original df shape: {df.shape}")
print(f"df_train shape: {df_train.shape}, df_val shape: {df_val.shape}, df_test shape: {df_test.shape}")
assert df_train.shape[0]+df_val.shape[0]+df_test.shape[0]==df.shape[0]

# Inspect data
print(f"Example text length: {len(df.text[0])}")
print(f"Example text: {df.text[0]}")
display(df.head(2))

original df shape: (100, 4)
df_train shape: (100, 4), df_val shape: (0, 4), df_test shape: (0, 4)
Example text length: 1113
Example text: The whole town of Blackstone is afraid, because they lynched Bret Dixon's brother - and he is coming back for revenge! At least that's what they think.<br /><br />A great Johnny Hallyday and a very interesting, early Mario Adorf star in this Italo-Western, obviously filmed in the Alps.<br /><br />Bret Dixon is coming back to Blackstone to investigate why his brother was lynched. He is a loner and gunslinger par excellance, everybody is afraid of him - the Mexican bandits (fighting the Gringos that took their land!) as well as the "decent" citizens that lynched Bret's brother. They lynched him, because they thought he stole their money instead of bringing it to Dallas to the safety of the bank there. But this is is only half the truth, as we find out in the course of this psychologically interesting western.<br /><br />But beware, it's kind of a depre

,movie_index,text,label_int,label
0,80,"The whole town of Blackstone is afraid, becaus...",1,Positive
1,84,This Harold Lloyd short wasn't really much; no...,0,Negative


In [12]:
# 检查每个类别的训练样本数量是否大致相等
class1=df_train.loc[(df_train.label=="Positive"),:].copy() # ':'保留这张表里的所有列
class2=df_train.loc[(df_train.label=="Negative"),:].copy()
print(f"Count samples positive: {class1.shape[0]}")
print(f"Count samples negative: {class2.shape[0]}")

Count samples positive: 50
Count samples negative: 50


## Chunking

在嵌入之前，需要先确定你的分块策略、分块大小和分块重叠度。在此演示中，我将采用以下设置：

- **Strategy**策略 = 除非评论过长，否则将电影评论作为单个分块处理。
- **Chunk size**分块大小 = 使用嵌入模型参数 MAX_SEQ_LENGTH
- **Overlap**重叠度 = 一般建议10%-15%
- **Function**函数 = 使用Langchain提供的便捷递归字符文本拆分器（RecursiveCharacterTextSplitter）对较长的评论进行递归拆分。

⚠️ **演示批次大小 = 100行，仅作演示用途。**

这意味着，拥有更多数据时，问题结果可能会更好！

### Exercise #2
修改 chunk_size 看会发生什么？模型的默认值是 511。

- 你的观察结果暗示了什么关于修改 chunk_size 和向量数量的关系？
    - chunk_size 变小，向量数量暴增，存储与速度开销变大。
    - chunk_size 变大，向量数量骤减，有截断风险（超过了模型的 511 限制，会导致模型自动截断末尾文本，丢失结尾信息）
- 当 chunk_size=256 时，有多少个向量？
    - 向量总数 ≈ 数据集总字符数 / (chunk_size × 0.9)

In [13]:
# Default chunk_size and overlap are calculated from embedding model parameters
chunk_size= 511
chunk_overlap=np.round(chunk_size*0.1,0)
BATCH_SIZE=100

# Chunk a batch of data from pandas DataFrame and inspect it
batch=_utils.imdb_chunk_test(encoder,BATCH_SIZE,df,chunk_size,chunk_overlap)

chunk_size:511
original shape:(100, 4)
chunk shape:(290, 5)
Chunking + embeddings time for 100 docs: 12.335254907608032 sec


,movie_index,text,chunk,vector,label_int,label
0,80,"The whole town of Blackstone is afraid, becaus...","The whole town of Blackstone is afraid, becaus...","[0.046037044, 0.034893896, 0.014340942, 0.0577...",1,Positive
1,80,"The whole town of Blackstone is afraid, becaus...",Mexican bandits (fighting the Gringos that too...,"[0.04041476, 0.013738137, -0.009600239, 0.0316...",1,Positive
2,80,"The whole town of Blackstone is afraid, becaus...",and definitely everybody is bad to the bone......,"[0.048768364, 0.012157818, -0.02666252, 0.0191...",1,Positive
3,84,This Harold Lloyd short wasn't really much; no...,This Harold Lloyd short wasn't really much; no...,"[0.03818898, 0.018209843, 0.037283037, 0.02589...",0,Negative
4,84,This Harold Lloyd short wasn't really much; no...,part was the last four or five minutes when th...,"[0.058892794, 0.019458644, 0.011418912, 0.0138...",0,Negative


type embeddings: <class 'pandas.Series'> of <class 'numpy.ndarray'>
of numbers: <class 'numpy.float32'>


## Insert data into Milvus

对于每个原始文本片段，我们将把四元组（`vector, text, source, h1, h2`）写入数据库。

Milvus 客户端封装器仅能处理从字典列表中加载数据。

否则，Milvus 通常支持从以下格式加载数据：

- pandas 数据框
- 字典列表

下面我们将使用 HuggingFace 提供的嵌入模型，下载其检查点，并在本地运行以作为编码器。

### STEP 5. INSERT CHUNKS AND EMBEDDINGS IN Milvus Lite

In [14]:
# Convert DataFrame to a list of dictionaries
dict_list=[]
for _,row in batch.iterrows():
    dictionary=row.to_dict()
    dict_list.append(dictionary)

print("Start inserting entities")
start_time=time.time()
insert_result=mc.insert(
    COLLECTION_NAME,
    data=dict_list,
    progress_bar=True,
)
end_time=time.time()
print(f"Milvus insert time for {batch.shape[0]} vectors: {end_time-start_time} sec")

# After final entity is inserted, call flush to stop growing segments left in memory
mc.flush(COLLECTION_NAME)

Start inserting entities
Milvus insert time for 290 vectors: 0.08771800994873047 sec


## Run a Semantic Search

现在我们可以对所有电影评论的嵌入向量进行快速搜索，找出与用户查询最接近的前K个电影评论。

- 在这个示例中，我们将为一位医生寻找电影推荐。

💡 为了保持一致性，所有嵌入向量都应使用相同的模型。

## Ask a question about your data

到目前为止，在这个演示笔记本中：

1. 您的自定义数据已被映射到向量嵌入空间
2. 这些向量嵌入已保存至向量数据库

接下来，您可以针对您的自定义数据提出问题！

💡 使用大语言模型时：

> **Query查询**是用户提问的统称。
> 一个查询可以包含多个独立问题，最多可达约1000个不同问题！

> **Question问题**通常指单个用户提出的问题。
> 在下面的例子中，用户的问题是：“我是一名医生，应该看哪部电影？”

In [15]:
# Define a sample question about your data
question="I'm a medical doctor, what movie should I watch"
query=[question]

# Inspect the length of quety
QUERY_LENGTH=len(query[0])
print(f"Query length: {QUERY_LENGTH}")

Query length: 47


**使用与之前相同的嵌入模型来嵌入问题**

为了使向量搜索能够正常工作，问题本身必须使用与您要搜索的集合创建时相同的模型进行嵌入

In [16]:
#Embed the query using same embedding model used to create the Milvus collection
query_embeddings=_utils.embed_query(encoder,query)

# Inspect data
print(type(query_embeddings),len(query_embeddings),type(query_embeddings[0]))
print(type(query_embeddings[0][0]))

<class 'list'> 1 <class 'numpy.ndarray'>
<class 'numpy.float32'>


## Execute a vector search

使用 [PyMilvus API](https://milvus.io/docs/zh/search.md) 搜索 Milvus。

💡 向量搜索本质上是“语义”搜索。例如，如果你搜索“leaky faucet(漏水的水龙头)”：

> **传统关键词搜索**——无论是“leaky”还是“faucet”，或者两者都必须与文本匹配，才能返回网页或文档链接。

> **语义搜索**——即使词不相同，但包含“drippy”、“taps”等词语的结果也会被返回，因为这些词具有相同的含义。

### Exercise #3

使用默认搜索索引搜索 Milvus

In [17]:
# Run semantic(语义) vector search using your query and the vector database

# Not Needed with Milvus Client API
# mc.load()

# Uses default search algorithm: HNSW and top_k=10
start_time=time.time()
results=mc.search(
    COLLECTION_NAME,
    data=query_embeddings
)
elaped_time=time.time()-start_time
print(f"Search time: {elaped_time} sec")

# Inspect search data
print(f"type: {type(results)}, count: {len(results[0])}")

Search time: 0.007001399993896484 sec
type: <class 'pymilvus.client.search_result.SearchResult'>, count: 10


In [18]:
# Re-run the search using custom settings

# Return top k results with HNSW index
TOP_K=3
OUTPUT_FIELDS=["movie_index","chunk","label"]
SEARCH_PARAMS=dict({
    # Re-use index params from num_candidate_nearest_neighbors
    "ef":INDEX_PARAMS['efConstruction']
})

# Run the search and time it
start_time = time.time()
results=mc.search(
    COLLECTION_NAME,
    data=query_embeddings,
    search_params=SEARCH_PARAMS,
    output_fields=OUTPUT_FIELDS,
    # Milvus 可以在布尔表达式中使用元数据来过滤搜索
    # expr=''
    limit=TOP_K,
    consistency_level="Eventually",
)

elaped_time=time.time()-start_time
print(f"Milvus Search time: {elaped_time} sec")

# Inspect search result
print(f"type: {type(results)}, count: {len(results[0])}")

Milvus Search time: 0.003002166748046875 sec
type: <class 'pymilvus.client.search_result.SearchResult'>, count: 3


## Assemble and inspect the search result

搜索结果存储在类型为 `'pymilvus.orm.search.SearchResult'` 的变量 `result[0]` 中

In [22]:
# 组装 `num_shot_answers` 个已获取的第1个上下文和上下文元数据
METADATA_FIELDS=[f for f in OUTPUT_FIELDS if f != 'chunk']
formatted_result,context,context_metadata=_utils.client_assemble_retrieved_context(
    results,
    metadata_fields=METADATA_FIELDS,
    num_shot_answers=3
)
print(f"Length context: {len(context[0])}, Number of contexts: {len(context)}")

# 循环遍历每个上下文和元数据并打印
for i in range(len(context)):
    print(f"Retrieved result #{i+1}")
    print(f"Context: {context[i][:150]}")
    print(f"Metadata: {context_metadata[i]}")
    print()

Length context: 507, Number of contexts: 3
Retrieved result #1
Context: Dr. K(David H Hickey)has been trying to master a formula that would end all disease and handicaps, but needs live donors to complete his work. His doc
Metadata: {'movie_index': '56', 'label': 'Negative'}

Retrieved result #2
Context: This movie took the Jerry Springer approach to super-human power. "Wilder Napalm" is the kind of theme-based movie that I love, addressing the idea th
Metadata: {'movie_index': '88', 'label': 'Positive'}

Retrieved result #3
Context: is not a horror movie, although it does contain some violent scenes, but is rather a comedy. A satire to be precise. And it never runs out of steam! T
Metadata: {'movie_index': '44', 'label': 'Positive'}



## Same question, but add Metadata filter

保持相同的问题，但在元数据上添加一个类似 SQL 的筛选条件。

我们期望得到与上述相同的结果，但排除所有标有“负面”标签的电影。

In [23]:
# Same question, but add Metadata filter only positive movies
metadata_filter="(label like 'Positive%')"

# Run the search and time it
start_time = time.time()
new_results=mc.search(
    COLLECTION_NAME,
    data=query_embeddings,
    search_params=SEARCH_PARAMS,
    output_fields=OUTPUT_FIELDS,
    filter=metadata_filter,
    limit=TOP_K,
    consistency_level="Eventually",
)

elaped_time=time.time()-start_time
print(f"Milvus Search time: {elaped_time} sec")

# Assemble `num_shot_answers` retrieved 1st context and context metadata
METADATA_FIELDS=[f for f in OUTPUT_FIELDS if f != 'chunk']
formatted_result,context,context_metadata=_utils.client_assemble_retrieved_context(
    new_results,
    metadata_fields=METADATA_FIELDS,
    num_shot_answers=3
)
print(f"Length context: {len(context[0])}, Number of contexts: {len(context)}")

# loop throught each context and metadata and print
for i in range(len(context)):
    print(f"Retrieved result #{i+1}")
    print(f"Context: {context[i][:150]}")
    print(f"Metadata: {context_metadata[i]}")
    print()

# As expected, same answers, except 'Negative' movies are omitted.

Milvus Search time: 0.018923044204711914 sec
Length context: 446, Number of contexts: 3
Retrieved result #1
Context: This movie took the Jerry Springer approach to super-human power. "Wilder Napalm" is the kind of theme-based movie that I love, addressing the idea th
Metadata: {'movie_index': '88', 'label': 'Positive'}

Retrieved result #2
Context: is not a horror movie, although it does contain some violent scenes, but is rather a comedy. A satire to be precise. And it never runs out of steam! T
Metadata: {'movie_index': '44', 'label': 'Positive'}

Retrieved result #3
Context: a good movie with a real good story. The fact that there are so many other big stars who all also had great performances is just an added BONUS! So do
Metadata: {'movie_index': '67', 'label': 'Positive'}



## Try another question

这次只需在问题中加上**only good movies**，看看答案是否有所不同？

对于语义不同的问题，我们预期答案会不同。

In [24]:
# Take as input a user question and conduct semantic vector search using the question
question="I'm a medical doctor, what movie should I watch?"
new_question="I'm a computer scientist, what movie should I watch?"
print(f"Question: {new_question}")

# Embed the query using same embedding model used to create the Milvus collection
new_query_embeddings=_utils.embed_query(encoder,[new_question])

# Run the search and time it
start_time=time.time()
new_results=mc.search(
    COLLECTION_NAME,
    data=new_query_embeddings,
    search_params=SEARCH_PARAMS,
    output_fields=OUTPUT_FIELDS,
    # Milvus can utilize metadata in boolean expressions to filter search.
    # expr=""
    limit=TOP_K,
    consistency_level="Eventually",
)

elaped_time=time.time()-start_time
print(f"Milvus Search time: {elaped_time} sec")

# Assemble `num_shot_answers` retrieved 1st context and context metadata
METADATA_FIELDS=[f for f in OUTPUT_FIELDS if f != 'chunk']
formatted_result,context,context_metadata=_utils.client_assemble_retrieved_context(
    new_results,
    metadata_fields=METADATA_FIELDS,
    num_shot_answers=3
)
print(f"Length context: {len(context[0])}, Number of contexts: {len(context)}")

# loop throught each context and metadata and print
for i in range(len(context)):
    print(f"Retrieved result #{i+1}")
    print(f"Context: {context[i][:150]}")
    print(f"Metadata: {context_metadata[i]}")
    print()

Question: I'm a computer scientist, what movie should I watch?
Milvus Search time: 0.0040013790130615234 sec
Length context: 446, Number of contexts: 3
Retrieved result #1
Context: This movie took the Jerry Springer approach to super-human power. "Wilder Napalm" is the kind of theme-based movie that I love, addressing the idea th
Metadata: {'movie_index': '88', 'label': 'Positive'}

Retrieved result #2
Context: Bears about as much resemblance to Dean Koontz's novel as Jessica Simpson does to a rocket scientist. If you've read the book, I suggest you put it as
Metadata: {'movie_index': '21', 'label': 'Positive'}

Retrieved result #3
Context: i would be curious what kids think of this movie. Maybe they would enjoy it? But as for adults, safe bet they wont, even if a CS fan.
Metadata: {'movie_index': '37', 'label': 'Negative'}



In [25]:
# Drop collection
client.drop_collection(COLLECTION_NAME)

In [ ]:
# 